<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os

REPO_URL = 'https://github.com/AhmedMahmoud-123/FlyRank_AI.git'
REPO_DIR = 'FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/FlyRank_AI


In [ ]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv


In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')
df.head()

30,000 rows loaded


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
"A page is worth reviewing if it used to get real traffic, its position is slipping, and it hasn't been touched in a while."

"A page is worth reviewing if it used to get real traffic, its position is slipping, and it hasn't been touched in a while."

In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows loaded')

# sanity check the rule's inputs exist and are in the ranges the dictionary promises
print(df[['impressions_prev_30d', 'avg_position', 'days_since_last_update']].describe())

30,000 rows loaded
       impressions_prev_30d  avg_position  days_since_last_update
count          30000.000000   30000.00000            30000.000000
mean            1783.078500      16.34238               46.098300
std             6150.429511      15.21679               42.078709
min                0.000000       0.00000                1.000000
25%               19.000000       6.20000               20.000000
50%              210.000000      10.80000               20.000000
75%             1143.000000      22.30000              104.000000
max           218786.000000     245.00000              373.000000


**Reasons:**
- `stale_but_visible` — real prior traffic, no recent update
- `position_slipping` — average position got worse last30 vs prev30
- `low_visibility` — doesn't clear the visibility bar; lowest-priority bucket

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np
from pathlib import Path

visible = (df['impressions_prev_30d'] >= 500).astype(int)
stale = (df['days_since_last_update'] >= 180).astype(int)
# avg_position=0 means "no data", not rank zero — exclude those rows from the slipping check
has_position = df['avg_position'] > 0
slipping = (has_position & (df['avg_position'] > df['avg_position'].median())).astype(int)

df['score'] = visible * (stale + slipping) * df['impressions_prev_30d']  # readable on purpose

def reason_code(row):
    if row['stale'] and row['visible']:
        return 'stale_but_visible'
    if row['slipping']:
        return 'position_slipping'
    return 'low_visibility'

df['visible'], df['stale'], df['slipping'] = visible, stale, slipping
df['reason_code'] = df.apply(reason_code, axis=1)

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

Path('work/outputs').mkdir(parents=True, exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(ranked):,} ranked rows to work/outputs/baseline_action_score.csv")
ranked[['content_id', 'score', 'reason_code']].head(10)

Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv


,content_id,score,reason_code
0,content_2cb567c3c89b,160641,position_slipping
1,content_2dba2b1f9536,137909,position_slipping
2,content_b28d1efd668f,110679,position_slipping
3,content_ff94c9b6b411,96122,position_slipping
4,content_813e88069237,94762,position_slipping
5,content_b511d4bc4ad2,83271,position_slipping
6,content_66b4046cc144,69303,position_slipping
7,content_05e9b4cd9ccf,69282,position_slipping
8,content_40fb6f005d61,66481,position_slipping
9,content_c5063073d048,65696,position_slipping


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked.head(20)[['content_id', 'score', 'reason_code',
                          'impressions_prev_30d', 'avg_position', 'days_since_last_update',
                          'is_declining_label']]
top20

,content_id,score,reason_code,impressions_prev_30d,avg_position,days_since_last_update,is_declining_label
0,content_2cb567c3c89b,160641,position_slipping,160641,22.2,48,0
1,content_2dba2b1f9536,137909,position_slipping,137909,27.9,104,0
2,content_b28d1efd668f,110679,position_slipping,110679,26.2,104,0
3,content_ff94c9b6b411,96122,position_slipping,96122,27.4,20,1
4,content_813e88069237,94762,position_slipping,94762,26.2,104,1
5,content_b511d4bc4ad2,83271,position_slipping,83271,27.9,104,0
6,content_66b4046cc144,69303,position_slipping,69303,26.6,20,1
7,content_05e9b4cd9ccf,69282,position_slipping,69282,22.1,104,1
8,content_40fb6f005d61,66481,position_slipping,66481,26.0,104,1
9,content_c5063073d048,65696,position_slipping,65696,12.5,104,0


## 3. Top-20 review

The rule surfaces mostly `position_slipping` picks, but this reason code is only a median-position proxy, not a true position trend. Therefore it should be treated as a weak heuristic rather than evidence that the page's position is actually declining.

**Confirmed hits** (`is_declining_label` = 1) — rows 3, 4, 6, 7, 8, 12, 13, 14, 16, 18, 19.
These are cases where the heuristic happened to align with the observed decline label; the rule itself does not prove that the position was declining.: real position weakness or staleness lining up with an
actual decline. Row 13 (`content_cf56e2e2e282`) is the cleanest case — 194 days untouched,
confirmed declining, exactly the `stale_but_visible` pattern the rule was written to catch.
Row 18 (`content_54baba704595`, position 47.0) and row 14 (position 32.0) are the weakest
positions in the whole list, so their high scores make sense.

**Weak picks** (`is_declining_label` = 0) — rows 0, 1, 2, 5, 9, 10, 11, 15, 17. Six of the
top 20 rows flagged `position_slipping` where nothing actually declined. Row 9
(`content_c5063073d048`) is the clearest miss: `avg_position` = 12.5, solidly page 1, flagged
only because 12.5 sits above the dataset *median* — "worse than average" got treated as
"declining," which isn't the same claim.

**Confidence note:** high confidence on the `stale_but_visible` hit and the two worst-position
hits; low confidence on any row where `position_slipping` fired but the label says no decline —
those need a real prev30-vs-last30 position delta, not a median comparison, before I'd trust them.

**What would flip these calls:** for the false positives, an actual position *trend* (last30 vs
prev30, not vs. the dataset median) would separate "consistently mid-pack" pages from pages that
are genuinely sliding.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining_label'].mean()
for k in [20, 50, 100]:
    p = precision_at_k(ranked['score'], ranked['is_declining_label'], k)
    print(f'precision@{k}: {p:.3f}  (base rate: {base_rate:.3f})')

# dummy floor: majority-class baseline
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(df[['impressions_prev_30d']], df['is_declining_label'])
print('dummy (majority class) accuracy:', dummy.score(df[['impressions_prev_30d']], df['is_declining_label']))

# leakage check — none of the score inputs may come from the label window or excluded fields
score_inputs = {'impressions_prev_30d', 'avg_position', 'days_since_last_update'}
label_window_fields = {'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'trend_pct', 'trend_direction'}
assert not (score_inputs & label_window_fields), "leakage: score uses a label-window field"
print("no label-window fields in the score — OK")

precision@20: 0.550  (base rate: 0.542)
precision@50: 0.600  (base rate: 0.542)
precision@100: 0.620  (base rate: 0.542)
dummy (majority class) accuracy: 0.5420666666666667
no label-window fields in the score — OK


These Precision@K values are descriptive baseline results calculated over the full available dataset. They are not a held-out generalization estimate; the later modeling notebook uses a client-grouped test split for model comparison.

## 4. Weak picks + leakage check

**The rule mostly collapsed to "sort by traffic."** For every row where `stale = 0`, the score formula reduces to `score = impressions_prev_30d` exactly — and the top-20 table confirms it: `score` and `impressions_prev_30d` match in 19 of 20 rows. `position_slipping` is on for almost every high-traffic page in this slice, so it is not adding much discrimination. Because traffic is multiplied into the score, the ranking is largely driven by `impressions_prev_30d` once a row passes the rule conditions.

**Concrete bad pick:** `content_c5063073d048` — flagged `position_slipping` at `avg_position`
12.5 (page 1), because the threshold compares to the dataset *median*, not to the page's own
prior position. `is_declining_label` = 0 confirms this shouldn't have ranked in the top 20.

**Worth double-checking:** `days_since_last_update = 104` appears repeatedly in the top-20 sample. Before relying heavily on this feature, I would check its overall value distribution to determine whether this reflects a real measurement pattern or a repeated/default value.

**Leakage check:** `score` is built only from `impressions_prev_30d`, `avg_position`, and
`days_since_last_update` — none of these touch the `last30`/label window, and none are on the
excluded list from the w03 data contract (`trend_direction`, `trend_pct`, `competition_level`,
etc.). The assert in the cell above confirms no overlap between score inputs and label-window
fields.

**Bottom line:** Precision@20 is 11/20 = 0.55, only slightly above the overall declining rate of 0.542. This means the baseline provides limited ranking lift at the top of the queue. The fix worth trying before ML-08 is replacing the median-position threshold with an
actual prev30-vs-last30 position delta, so `position_slipping` means "got worse" instead of
"worse than average."

In [ ]:
print(
    df['days_since_last_update']
    .value_counts()
    .head(15)
)

days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
14       505
28       466
25       441
7        420
11       246
15       231
6        183
26       143
19       129
1        118
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.